# 第 3 课：余弦相似度 + DTW 组合检索

目标：从经验库找出与当前机动最相似的历史案例，并完成 `alpha` 消融。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 3.1 构建查询和记忆库

核心源码：[retrieval.py](../src/memcast_uav/retrieval.py)  
文字讲解：[03_retrieval.md](../tutorial/03_retrieval.md)

In [ ]:
from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.features import extract_motion_features
from memcast_uav.memory import build_memory
from memcast_uav.retrieval import RetrievalConfig, retrieve

flight = make_synthetic_flight()
train, test = make_train_test_windows(flight, split_index=504)
memory = build_memory(train, limit=30)
query = test[0]
query_features = extract_motion_features(query.history, query.dt)
print("记忆条目:", len(memory.entries), "查询意图:", query.intent)

## 3.2 组合检索 Top-3

In [ ]:
config = RetrievalConfig(
    alpha=0.5,
    gamma=120.0,
    top_k=3,
    intent_weight=0.25,
)
ranked = retrieve(
    query.history,
    query_features,
    query.intent,
    memory,
    config,
)

for rank, item in enumerate(ranked, start=1):
    print(
        f"Top {rank}: {item.entry.entry_id}, "
        f"intent={item.entry.intent}, cos={item.feature_similarity:.4f}, "
        f"dtw={item.dtw_distance:.2f}, match={item.intent_match:.0f}, "
        f"score={item.final_score:.4f}"
    )

组合公式：

`base = alpha × cosine + (1-alpha) × exp(-DTW/gamma)`

`final = base + confidence_weight × confidence + intent_weight × intent_match`

## 3.3 分块练习：只改 alpha

In [ ]:
def top_ids(alpha: float) -> list[str]:
    results = retrieve(
        query.history,
        query_features,
        query.intent,
        memory,
        RetrievalConfig(
            alpha=alpha,
            gamma=120.0,
            top_k=3,
            intent_weight=0.25,
        ),
    )
    return [item.entry.entry_id for item in results]

print("alpha=0（只看 DTW）:", top_ids(0.0))
print("alpha=1（只看特征）:", top_ids(1.0))
# TODO：记录排名差异，并解释两种相似度分别捕捉什么。

## 3.4 本课验收

In [ ]:
scores = [item.final_score for item in ranked]
assert scores == sorted(scores, reverse=True)

import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_retrieval.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 2 课](02_features.ipynb) · [教程目录](README.md) · [下一课：经验记忆 →](04_memory.ipynb)